In [1]:
%pip install -qU aiosqlite langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv('GROQ_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [3]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict
import sqlite3

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        
        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_llm)

        graph.add_node("action", self.take_action)

        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_llm(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        print("Mensajes enviados al modelo:", messages)
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Llamando la Herramienta: {t['name']} con los siguientes argumentos: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Retornando a la LLM tras la acción!")
        return {'messages': results}

In [7]:
import os
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

# current_tavily_api_key = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY encontrado. Certifique-se de que está no seu .env e python-dotenv está instalado."
    )

tool = TavilySearch(
    max_results=3,
    tavily_api_key=TAVILY_API_KEY,
)

prompt_system = """
Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.
Tienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).
Busca información únicamente cuando tengas certeza de qué buscar.
Si necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.
Cuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.
"""

model = ChatGroq(
    model="llama-3.3-70b-versatile",  # También puedes usar otros modelos de Groq
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

abot = Agent(
    model,
    [tool],
    system=prompt_system,
    checkpointer=memory,
)

In [10]:
messages = [HumanMessage(content="Cómo está el clima santiago de chile el dia de hoy 24/07/26?")]
thread = {"configurable": {"thread_id": "3"}} 


print("\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='Cómo está el clima en Lima - Perú hoy (29 de diciembre de 2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p5ebjbwmk', 'function': {'arguments': '{"end_date":"2025-12-29","query":"Cl

In [12]:
messages = [HumanMessage(content="clima para coquimbo chile para hoy?")]
thread = {"configurable": {"thread_id": "1"}} 


print("\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='clima para coquimbo chile para hoy?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'j175kzg7c', 'function': {'arguments': '{"query":"Clima Coquimbo Chile hoy","search_depth":"basic",

In [13]:
messages = [HumanMessage(content="clima para copiado chile para hoy?")]
thread = {"configurable": {"thread_id": "2"}} 


print("\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='clima para copiado chile para hoy?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hfhyf95mx', 'function': {'arguments': '{"query":"clima para Copiado Chile hoy","search_depth":"basi

In [14]:
messages = [HumanMessage(content="clima para linares chile para hoy?")]
thread = {"configurable": {"thread_id": "4"}} 


print("\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='clima para linares chile para hoy?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'j32v06ccp', 'function': {'arguments': '{"query":"clima Linares Chile hoy","search_depth":"basic","t

In [15]:
messages = [HumanMessage(content="que region de chile fue la con menor temperatura?")]
thread = {"configurable": {"thread_id": "6"}} 


print("\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='que region de chile fue la con menor temperatura?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rfaay41g5', 'function': {'arguments': '{"query":"región de Chile con menor temperatu

In [16]:
messages = [HumanMessage(content="Cuál ciudad está más caliente?")]
thread = {"configurable": {"thread_id": "3"}} 

print("\n--- Pregunta 3: Comparación ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pregunta 3: Comparación ---
Mensajes enviados al modelo: [SystemMessage(content='\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='Cómo está el clima en Lima - Perú hoy (29 de diciembre de 2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p5ebjbwmk', 'function': {'arguments': '{"end_date":"2025-12-29","query":"Clim